<a href="https://colab.research.google.com/github/gt-cse-6040/bootcamp/blob/main/notebooks/sql/SQL3nb3_SQL_CTEs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SQL Common Table Expressions (CTE)

In [2]:
!wget https://github.com/gt-cse-6040/bootcamp/raw/main/assets/data/NYC-311-2M_small.db

'wget' is not recognized as an internal or external command,
operable program or batch file.


In [3]:
# create a connection to the database
import sqlite3 as db
import pandas as pd

# Connect to a database (or create one if it doesn't exist)
conn_nyc = db.connect('NYC-311-2M_small.db')

## Common Table Expressions (CTEs) -- Quick Review
- A Common Table Expression (CTE) is a named, temporary result set defined using the WITH clause. It's like a subquery that you define once and can reference multiple times.

- CTEs break complex queries into logical, named steps. This makes code much easier to read, debug, and maintain compared to deeply nested subqueries.

- They keep your code "DRY" (Don't Repeat Yourself). By defining the logic once at the top, you can join to it repeatedly, avoiding the performance overhead of re-calculating the same subquery.

- CTEs are ideal when you need to reference the same set of data multiple times, or when you need to perform complex aggregations alongside individual row-level data.

- A CTE exists only for the duration of the single query in which it is defined. It acts as a "temporary view" that disappears as soon as your main query finishes executing.

Visit the [SQLite Documentation's CTE page](https://www.sqlite.org/lang_with.html) to learn more.

### EXAMPLE -- INNER JOIN WITH CTE (same as before, with a subquery)

**Requirement**

From the `data` table, for each `city`, return the total counts or a distribution of tickets per `hour` on the biggest day (by events) by `createdDate`.

* Columns
    * `City`
    * `createdHour`
    * `countoccur`: the count of events

* Exclude NULL cities i.e. `WHERE city IS NOT NULL`

* Sort
    * `City` in ascending order
    * `createdHour` in ascending order

**Pseudocode:**
*    Need to find the biggest day
*    JOIN to the `data` table
*    produce `SELECT` statement
*    include `WHERE` statement
*    `GROUP BY`
*    `ORDER BY`

In [29]:
def inner_join_example():

#     # display the list of cities
    query_inner_join = '''
                WITH topymd AS (
                                  SELECT strftime('%Y-%m-%d',CreatedDate) createdymd
                                          ,count(*) totalymd
                                  FROM data
                                  GROUP BY createdymd
                                )
                SELECT a.city
                        ,strftime('%H',CreatedDate) createdhour
                        ,count(*) countoccur
                FROM data a
                INNER JOIN topymd b on strftime('%Y-%m-%d',a.CreatedDate) = b.createdymd AND b.totalymd = (SELECT MAX(totalymd) FROM topymd)

                WHERE a.city IS NOT NULL
                    
                GROUP BY a.city, createdymd, createdhour
                ORDER BY a.city, createdhour
                '''
    return query_inner_join

df_inner_join_example = pd.read_sql(inner_join_example(),conn_nyc)
display(df_inner_join_example)

,City,createdhour,countoccur
0,ARVERNE,11,1
1,ARVERNE,12,2
2,ASTORIA,03,1
3,ASTORIA,07,1
4,ASTORIA,10,2
...,...,...,...
623,Woodside,13,1
624,Woodside,15,1
625,Woodside,16,1
626,Woodside,17,1


### So what did we do here?

- The CTE counted the number of rows (complaints) for each date.

- The CTE inner is joined to the main query on date and the date(s) with the largest count, which "filters" the results to only the covers the event of ties.

### LEFT JOIN WITH SUBQUERY AND MULTIPLE CTEs

From the `data` table, for each `city`, return counts or a distribution of tickets per `hour` on the biggest day (by events) by `createdDate`.

Recall it's `2014-11-18` (8466 events) but how do we put this into code dynamically?

But this time, **only include `City` if they have an event on that day**.

Additionally, `City` with `Hour` without events on that day should be included but should have a 0 for `countoccur`.

* Columns
    * `City`
    * `createdHour`
    * `countoccur`: the count of events. **REMEMBER: This should be 0 if there aren't any events for that city/hour**


* Exclude NULL cities i.e. `WHERE city IS NOT NULL`


* Sort
    * `City` in ascending order
    * `createdHour` in ascending order

**Pseudocode:**

*    Need to find the biggest day.

*    Need to get all the hours that occur on the biggest day. 

*    Need to get all the cities/hour combinations possible for the biggest day. 

*    Need to get the counts for each city/hour on the biggest day.

*    Need to join the combinations with the counts. 

*    produce `SELECT` statement

*    `GROUP BY`
*    `ORDER BY`

### Note that there are 5 distinct queries/subqueries:

- CTE_Counts gets our total count by day
- CTE_hours creates an hour row for each hour of each day
- CTE_cityhours joins the previous CTEs together and filters for the top day
- CTE_cityhourscounts does the actual aggregation
- Main query joins CTE_cityhours and CTE_cityhourscounts together 

In [28]:
def left_join_example():
    query_left_join = '''
                            -- Step 1: Calculate daily counts for all days
                            WITH CTE_Counts as ( 
                                
                                SELECT strftime('%Y-%m-%d', CreatedDate) AS createdymd,
                                    count(*) AS totalymd
                                FROM data
                                GROUP BY strftime('%Y-%m-%d', CreatedDate)
                                ),
                            --Step 2: Creates the distinct hours 
                                CTE_hours AS (
                                          SELECT distinct strftime('%Y-%m-%d',CreatedDate) createdymd
                                                          ,strftime('%H',CreatedDate) createdhour
                                          FROM data
                                        ),
                            -- Step 3: Merge the counts and the hours together
                                CTE_cityhours AS ( 
                                    SELECT DISTINCT a.city,
                                                    c.createdhour,
                                                    c.createdymd
                                    FROM data a
                                    INNER JOIN CTE_Counts b 
                                        ON strftime('%Y-%m-%d', a.CreatedDate) = b.createdymd
                                    INNER JOIN CTE_hours c 
                                        ON strftime('%Y-%m-%d', a.CreatedDate) = c.createdymd
                                    WHERE a.city IS NOT NULL AND b.totalymd = (SELECT MAX(totalymd) FROM CTE_Counts)
                                ),
                            -- Step 4: Aggregate actual counts for the busiest day(s)
                                CTE_cityhourcounts AS ( 
                                    SELECT a.city,
                                        strftime('%Y-%m-%d', a.CreatedDate) AS createdymd,
                                        strftime('%H', a.CreatedDate) AS createdhour,
                                        count(*) AS countoccur
                                    FROM data a
                                    INNER JOIN CTE_Counts b 
                                        ON strftime('%Y-%m-%d', a.CreatedDate) = b.createdymd
                                    WHERE a.city IS NOT NULL
                                    AND b.totalymd = (SELECT MAX(totalymd) FROM CTE_Counts)
                                    GROUP BY a.city, createdymd, createdhour
                                )
                            -- Main Query: Combine step 3 and step 4
                                SELECT aa.city,
                                    aa.createdymd,
                                    aa.createdhour,
                                    COALESCE(bb.countoccur, 0) AS countoccur
                                FROM CTE_cityhours aa
                                LEFT JOIN CTE_cityhourcounts bb
                                    ON aa.city = bb.city
                                AND aa.createdymd = bb.createdymd
                                AND aa.createdhour = bb.createdhour
                                ORDER BY aa.createdymd, aa.city, aa.createdhour;
                '''
    return query_left_join

df_left_join_example = pd.read_sql(left_join_example(),conn_nyc)
display(df_left_join_example)

,city,createdymd,createdhour,countoccur
0,ARVERNE,2014-11-18,00,0
1,ARVERNE,2014-11-18,01,0
2,ARVERNE,2014-11-18,02,0
3,ARVERNE,2014-11-18,03,0
4,ARVERNE,2014-11-18,04,0
...,...,...,...,...
2107,Woodside,2014-11-18,19,1
2108,Woodside,2014-11-18,20,0
2109,Woodside,2014-11-18,21,0
2110,Woodside,2014-11-18,22,0
